






## Webpage Extraction and Embedding (PolyU CUS)

### 1. Extracting raw text data

In [1]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader, SitemapLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

USER_AGENT environment variable not set, consider setting it to identify your requests.
C:\Users\Marcus\AppData\Local\Temp\ipykernel_12012\422639125.py:13: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!


In [2]:
cus_URL = "https://www.polyu.edu.hk/cus/"
docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "html.parser")   # Strip the html syntax
    
    # Get the title of the page
    title = soup.title.string if soup.title else "No Title"

    # Remove script and style elements
    for unwanted in soup(["script", "style"]):
        unwanted.decompose()
    
    # Define common navigational/junk tags to remove
    unwanted_tags = ['header', 'footer', 'nav', 'aside', 'button', 'form', 'noscript']
    
    # Attempt to find main content areas
    main_content = soup.find('main') or soup.find('article') or soup.find('div', class_='content')
    
    if main_content:
        for unwanted in main_content.find_all(unwanted_tags):
            unwanted.decompose()
        text = re.sub(r"\n\n+", "\n\n", main_content.get_text()).strip()
    else:
        for unwanted in soup.find_all(unwanted_tags):
            unwanted.decompose()
        text = re.sub(r"\n\n+", "\n\n", soup.text).strip()
    
    # Post-processing to remove lines that look like menu items
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Skip lines that are very likely menu navigation
        if re.search(r'menu|Search|Contact Us|Sitemap', line, re.I):
            continue
        cleaned_lines.append(line)
    
    text = "\n".join(cleaned_lines)

    if len(text) < 200:
        # Check if there's an actual paragraph or significant text block
        paraagraph = soup.find_all('p')
        length = sum(len(p.get_text()) for p in paraagraph)
        if length < 100:
            return f"{title}\n\n[No significant text content]"

    return f"{title}\n\n{text}"

cus_loader = RecursiveUrlLoader(
    max_depth=6,
    url=cus_URL,
    base_url=cus_URL,
    prevent_outside=True,
    exclude_dirs=[
        cus_URL+"about-ous",
        cus_URL+"about-cus",
        cus_URL+"Sitemap", 
        cus_URL+"sitemap",
        cus_URL+"Search-Result", 
        cus_URL+"search-result", 
        cus_URL+"internal",
        cus_URL+"docdrive",
        cus_URL+"-",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = cus_loader.lazy_load()
for doc in docs_lazy:
    print(doc.metadata.get('source'))
    for key in unwanted_metadata:
        if key in doc.metadata:
            del doc.metadata[key]
    docs.append(doc)

https://www.polyu.edu.hk/cus/
https://www.polyu.edu.hk/cus/undergraduate-studies-support/staff/academic-advising-at-polyu/
https://www.polyu.edu.hk/cus/student/
https://www.polyu.edu.hk/cus/study/minor/minor-in-research-and-innovation/
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/credit-transfer/
https://www.polyu.edu.hk/cus/student/senior-year-intakes-and-articulation-degree-programme/credit-transfer/
https://www.polyu.edu.hk/cus/undergraduate-studies-support/student/thank-you-note/
https://www.polyu.edu.hk/cus/study/minor/
https://www.polyu.edu.hk/cus/ielts/
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/second-major/
https://www.polyu.edu.hk/cus/undergraduate-studies-support/student/academic-advising/
https://www.polyu.edu.hk/cus/student/senior-year-intakes-and-articulation-degree-programme/general-university-requirements/
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/general-university-requirements/
https://www.polyu.edu.h

In [3]:
idx = 15
print(f"Extracted number of webpages in CUS: {len(docs)}")
print(docs[idx].page_content)
print(docs[idx].metadata.get('source'))

'''
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in CUS: 71
Admission | College of Undergraduate Studies

Please get the latest admission info here.
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/admission/


"\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1) if re.search(r'/([^/]+)/?$', source) else "unknown"
    chunk.metadata["chunk_id"] = f"PolyU_CUS_{url_end}_chunk_{i}"
    
    chunk.page_content = f"--- PolyU College of Undergraduate Studies (CUS) Website ---\n\n{chunk.page_content}"

print("Number of chunks: ", len(chunks))

Number of chunks:  255


In [5]:
print(chunks[20])

page_content='--- PolyU College of Undergraduate Studies (CUS) Website ---

Credit Transfer | College of Undergraduate Studies' metadata={'source': 'https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/credit-transfer/', 'content_type': 'text/html; charset=utf-8', 'title': 'Credit Transfer | College of Undergraduate Studies', 'description': 'Students should submit an application for credit transfer upon your initial enrolment on the programme or before the end of the add/drop period of the f...', 'chunk_id': 'PolyU_CUS_credit-transfer_chunk_20'}


### 3. Document Embedding in Chroma

In [6]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

In [7]:
client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

991

In [8]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

In [9]:
for i, chunk in enumerate(chunks):
    print(f"Adding chunk {i+1}/{len(chunks)} to ChromaDB. Metadata: {chunk.metadata}\n")
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i + 3000)]
    )

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Adding chunk 1/255 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/cus/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | College of Undergraduate Studies', 'chunk_id': 'PolyU_CUS_cus_chunk_0'}

Adding chunk 2/255 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/cus/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | College of Undergraduate Studies', 'chunk_id': 'PolyU_CUS_cus_chunk_1'}

Adding chunk 3/255 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/cus/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | College of Undergraduate Studies', 'chunk_id': 'PolyU_CUS_cus_chunk_2'}

Adding chunk 4/255 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/cus/undergraduate-studies-support/staff/academic-advising-at-polyu/', 'content_type': 'text/html; charset=utf-8', 'title': 'Academic Advising at PolyU | College of Undergraduate Studies', 'chunk_id': 'PolyU_CUS_academic-advising-at-polyu_chunk_3'}

Adding chunk

### 4. Simple Testing

In [10]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "What is the general university requirement for undergraduate student?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Source: {result.metadata.get('info')}")
    print(f"Content: {result.page_content}...")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

C:\Users\Marcus\AppData\Local\Temp\ipykernel_12012\1956528359.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


Source: None
Content: --- PolyU College of Undergraduate Studies (CUS) Website ---

General University Requirements (GUR) | College of Undergraduate Studies

GUR for 4-Year Undergraduate Student
Freshman Seminar
Language & Communication Requirements
Leadership & Intra-Personal Development
Cluster-Area Requirements
Service-Learning
Healthy Lifestyle
Artificial Intelligence and Data Analytics Requirement
Innovation and Entrepreneurship Requirement
Language & Communication Requirements
Leadership Education and Development
Cluster-Area Requirements
Service-Learning
Healthy Lifestyle
For the details of curriculum framework of the General University Requirements (GUR), please click here....
Chunk ID: PolyU_CUS_general-university-requirements_chunk_50

Source: None
Content: --- PolyU College of Undergraduate Studies (CUS) Website ---

General University Requirements (GUR) | College of Undergraduate Studies

GUR for Senior Year Intakes and Articulation Degree Programme
Language & Communication